# 03 · Queries analíticas de validación — TodoComponentes

En este notebook se:
1. Autentica contra BigQuery (mismo patrón que el notebook 02).
2. Generan 5 queries de validación contra distintas tablas.
3. Visualizan los resultados de cada query.

In [1]:
# --- 1. Autenticación y cliente BigQuery ---
import os
from pathlib import Path

from dotenv import load_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account


def _find_project_root() -> Path:
    for cand in (Path.cwd(), *Path.cwd().parents):
        if (cand / ".env").exists() or (cand / ".env.example").exists():
            return cand
    return Path.cwd().parents[2]


PROJECT_ROOT = _find_project_root()
load_dotenv(PROJECT_ROOT / ".env")

PROJECT_ID = os.environ["GCP_PROJECT_ID"]
DATASET_ID = os.environ["BQ_DATASET_ID"]
CREDENTIALS_PATH = str((PROJECT_ROOT / os.environ["GOOGLE_APPLICATION_CREDENTIALS"]).resolve())
assert os.path.exists(CREDENTIALS_PATH), f"Credenciales no encontradas en: {CREDENTIALS_PATH}"

credentials = service_account.Credentials.from_service_account_file(
    CREDENTIALS_PATH, scopes=["https://www.googleapis.com/auth/bigquery",
                              "https://www.googleapis.com/auth/bigquery.readonly"]
)

client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location="EU")

Q = f"{PROJECT_ID}.{DATASET_ID}"

def run_query(sql):
    return client.query(sql).to_dataframe()

In [2]:
# --- 2. Top 10 productos más vendidos ---
query_top_productos = f"""
SELECT
  oi.product_id,
  p.name AS product_name,
  SUM(oi.quantity) AS units_sold
FROM `{Q}.order_items` oi
JOIN `{Q}.products` p
  ON oi.product_id = p.product_id
GROUP BY product_id, product_name
ORDER BY units_sold DESC
LIMIT 10;
"""

df_top_productos = run_query(query_top_productos)
df_top_productos

c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,product_id,product_name,units_sold
0,62,ArduinoStore Arduino Nano v3,199
1,39,ElectroLab Teensy 4.1 v2,190
2,48,ElectroLab Regulador AMS1117-3.3 v1,186
3,19,CircuitPro Teensy 4.1 v1,186
4,9,SensorWorld Modulo WiFi ESP8266 v2,181
5,56,ElectroLab Buck 3A 12V-5V v3,171
6,11,ArduinoStore ATSAMD21 XPRO v1,171
7,53,TechKit Antena 2.4GHz IPEX v2,170
8,42,MakersHub Antena 2.4GHz IPEX v2,169
9,64,TechKit Modulo BLE nRF52840 v2,168


In [3]:
# --- 3. Ingresos por categoría ---
query_ingresos_categoria = f"""
SELECT
  c.category_id,
  c.name AS category_name,
  SUM(oi.line_total) AS revenue
FROM `{Q}.order_items` oi
JOIN `{Q}.products` p
  ON oi.product_id = p.product_id
JOIN `{Q}.categories` c
  ON p.category_id = c.category_id
GROUP BY category_id, category_name
ORDER BY revenue DESC;
"""

df_ingresos_categoria = run_query(query_ingresos_categoria)
df_ingresos_categoria


c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,category_id,category_name,revenue
0,6,Conectividad,114311.820000000
1,5,Energía y fuentes,111950.590000000
2,1,Microcontroladores,70617.470000000
3,4,Almacenamiento,58503.340000000
4,2,Sensores,43345.630000000
5,3,Display y HMI,40254.780000000


In [4]:
# --- 4. Clientes con mayor gasto total ---
query_clientes_gasto = f"""
SELECT
  cu.customer_id,
  cu.first_name,
  cu.last_name,
  SUM(oi.line_total) AS total_spent
FROM `{Q}.customers` cu
JOIN `{Q}.orders` o
  ON cu.customer_id = o.customer_id
JOIN `{Q}.order_items` oi
  ON o.order_id = oi.order_id
GROUP BY customer_id, first_name, last_name
ORDER BY total_spent DESC
LIMIT 20;
"""

df_clientes_gasto = run_query(query_clientes_gasto)
df_clientes_gasto


c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,first_name,last_name,total_spent
0,1,Tatiana,Amaral,36212.760000000
1,2,Geraldo,Gálvez,27669.350000000
2,3,Pauline,Söding,18444.010000000
3,5,Wiltrud,Kitzmann,15919.350000000
4,4,Daniela,Jover,14018.580000000
5,7,Felipe,Rodriguez,10069.590000000
6,6,Théophile,Antoine,8616.050000000
7,8,Pascual,Aller,7773.540000000
8,9,Valérie,Guyot,7017.240000000
9,16,Francisco,Reis,6279.220000000


In [5]:
# --- 5. Ventas mensuales + crecimiento porcentual ---
query_ventas_mensuales = f"""
WITH monthly_sales AS (
  SELECT
    FORMAT_DATE('%Y-%m', o.order_date) AS month,
    SUM(oi.line_total) AS revenue
  FROM `{Q}.orders` o
  JOIN `{Q}.order_items` oi
    ON o.order_id = oi.order_id
  GROUP BY month
)
SELECT
  month,
  revenue,
  LAG(revenue) OVER (ORDER BY month) AS prev_revenue,
  SAFE_DIVIDE(revenue - LAG(revenue) OVER (ORDER BY month),
              LAG(revenue) OVER (ORDER BY month)) * 100 AS growth_pct
FROM monthly_sales
ORDER BY month;
"""

df_ventas_mensuales = run_query(query_ventas_mensuales)
df_ventas_mensuales


c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,month,revenue,prev_revenue,growth_pct
0,2025-09,35411.840000000,None,None
1,2025-10,36864.630000000,35411.840000000,4.102554400
2,2025-11,29658.420000000,36864.630000000,-19.547761600
3,2025-12,34570.020000000,29658.420000000,16.560558500
4,2026-01,39933.070000000,34570.020000000,15.513586600
5,2026-02,33113.910000000,39933.070000000,-17.076473200
6,2026-03,40457.280000000,33113.910000000,22.176088500
7,2026-04,33793.470000000,40457.280000000,-16.471225900
8,2026-05,31113.870000000,33793.470000000,-7.929342600
9,2026-06,40928.700000000,31113.870000000,31.544870500


In [6]:
# --- 6. Tasa de conversión por método de pago ---
query_conversion_pago = f"""
SELECT
  p.payment_method,
  COUNT(*) AS total_payments,
  SUM(CASE WHEN p.payment_status = 'paid' THEN 1 ELSE 0 END) AS successful_payments,
  SAFE_DIVIDE(
    SUM(CASE WHEN p.payment_status = 'paid' THEN 1 ELSE 0 END),
    COUNT(*)
  ) AS conversion_rate
FROM `{Q}.payments` p
GROUP BY payment_method
ORDER BY conversion_rate DESC;
"""

df_conversion_pago = run_query(query_conversion_pago)
df_conversion_pago

c:\Users\ak471\Documents\Github\Team_Challenge_SQL\venv\Lib\site-packages\google\cloud\bigquery\table.py:2130: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,payment_method,total_payments,successful_payments,conversion_rate
0,paypal,389,307,0.789203
1,credit_card,411,320,0.778589
2,bank_transfer,402,312,0.776119
3,debit_card,415,322,0.775904
4,bizum,383,291,0.759791
